In [1]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
data = load_diabetes()
X, y = data['data'], data['target']
X.shape, y.shape

((442, 10), (442,))

In [3]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
# Standardize inputs and target
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_test = y_scaler.transform(y_test.reshape(-1, 1))

In [5]:
# Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [6]:
class AE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 5)
        self.fc2 = nn.Linear(5, 2)
        self.fc3 = nn.Linear(2, 5)
        self.fc4 = nn.Linear(5, input_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.fc4(x)
        
        return x, z

In [ ]:
class AE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 5)
        self.fc2 = nn.Linear(5, 2)
        self.fc3 = nn.Linear(2, 5)
        self.fc4 = nn.Linear(5, input_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        z = self.fc2(x)    # z is the embedding space
        x = self.fc3(z)
        x = self.relu(x)
        x = self.fc4(x)
        
        return x, z      # we also return z here. 
        
# So keep in mind that 
# X_hat, Z_hat = model(X_train)

In [31]:
class AE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 5)
        self.fc2 = nn.Linear(5, 2)
        self.fc3 = nn.Linear(2, 5)
        self.fc4 = nn.Linear(5, input_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        z = self.encoder(x)
        x = self.decoder(z)
        return x

    def encoder(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        z = self.fc2(x)
        return z

    def decoder(self, z):
        x = self.fc3(z)
        x = self.relu(x)
        x = self.fc4(x)
        return x

In [32]:
model = AE(input_dim=10)

In [33]:
# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

In [34]:
# Training loop
epochs = 50
model.train()

for epoch in range(epochs):

    X_hat = model(X_train)
    loss = criterion(X_hat, X_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}/{epochs} | Train MSE: {loss.item():.2f}")

Epoch 1/50 | Train MSE: 1.08
Epoch 2/50 | Train MSE: 1.07
Epoch 3/50 | Train MSE: 1.05
Epoch 4/50 | Train MSE: 1.04
Epoch 5/50 | Train MSE: 1.02
Epoch 6/50 | Train MSE: 1.01
Epoch 7/50 | Train MSE: 0.99
Epoch 8/50 | Train MSE: 0.97
Epoch 9/50 | Train MSE: 0.95
Epoch 10/50 | Train MSE: 0.93
Epoch 11/50 | Train MSE: 0.91
Epoch 12/50 | Train MSE: 0.89
Epoch 13/50 | Train MSE: 0.88
Epoch 14/50 | Train MSE: 0.86
Epoch 15/50 | Train MSE: 0.85
Epoch 16/50 | Train MSE: 0.84
Epoch 17/50 | Train MSE: 0.82
Epoch 18/50 | Train MSE: 0.81
Epoch 19/50 | Train MSE: 0.80
Epoch 20/50 | Train MSE: 0.79
Epoch 21/50 | Train MSE: 0.78
Epoch 22/50 | Train MSE: 0.77
Epoch 23/50 | Train MSE: 0.77
Epoch 24/50 | Train MSE: 0.76
Epoch 25/50 | Train MSE: 0.76
Epoch 26/50 | Train MSE: 0.75
Epoch 27/50 | Train MSE: 0.75
Epoch 28/50 | Train MSE: 0.74
Epoch 29/50 | Train MSE: 0.74
Epoch 30/50 | Train MSE: 0.73
Epoch 31/50 | Train MSE: 0.73
Epoch 32/50 | Train MSE: 0.73
Epoch 33/50 | Train MSE: 0.72
Epoch 34/50 | Train

In [36]:
Z = model.encoder(X_train)
Z.shape

torch.Size([353, 2])